# SDB Framework — run the whole pipeline in Jupyter (Python)

This notebook runs the **complete paper pipeline** (training → spatial CV →
bootstrap → uncertainty → GeoTIFF prediction → publication figures) using the
**Python API** of `sdb_framework_final.py`. No `.bat` / terminal needed.

> **Where to open this notebook:** it must live in the repository root
> (next to `sdb_framework_final.py`, `config.yaml` and `data/`).
>
> **Always runs with `config.yaml`** — the published paper source stays
> byte-for-byte unchanged; portability comes only from the config file.

In [ ]:
# 1) Install dependencies (first run only; ~a few minutes)
%pip install -q -r requirements.txt

In [ ]:
# 2) Make sure Jupyter is in the repository root and all inputs exist
from pathlib import Path
import os

if not Path("sdb_framework_final.py").exists():
    for candidate in [Path.cwd().parent, Path.home() / "SDB-Framework"]:
        if (candidate / "sdb_framework_final.py").exists():
            os.chdir(candidate)
            break

assert Path("sdb_framework_final.py").exists(), "Open this notebook from the repository root."
assert Path("config.yaml").exists(), "config.yaml is missing."
assert Path("data/SDBNew Latlong.xlsx").exists(), "Put 'SDBNew Latlong.xlsx' inside data/."
assert Path("data/20220213_SDB_5bands.tif").exists(), "Put '20220213_SDB_5bands.tif' inside data/."

print("Repository root :", Path.cwd())
print("Ready ✔")

## Run the complete pipeline

Equivalent to:
`python sdb_framework_final.py --mode full --config config.yaml`

Runtime depends on the machine (often 15–60 min for the full model
tournament with 10 CV repeats × 2000 bootstrap resamples). Progress is
printed below.

In [ ]:
from sdb_framework_final import SDBOperationalPipeline

pipeline = SDBOperationalPipeline(config_path="config.yaml")
artifacts = pipeline.run_full_pipeline()   # full pipeline, same as --mode full
print("Finished ✔  outputs written to:", pipeline.cfg.output_dir)

### Optional — quick smoke test instead of the full tournament

Comment out the cell above and run this one first if you just want to
verify that everything works end-to-end (a few minutes):

In [ ]:
# from sdb_framework_final import SDBConfig, SDBOperationalPipeline

# cfg = SDBConfig.from_yaml("config.yaml")
# cfg.n_repeats = 3            # fewer CV repetitions
# cfg.n_bootstrap_ci = 200     # fewer bootstrap resamples
# pipeline = SDBOperationalPipeline(cfg=cfg)
# artifacts = pipeline.run_full_pipeline(
#     model_candidates=["lyzenga", "random_forest"],
# )

## Key results

In [ ]:
from pathlib import Path

metrics = artifacts.get("holdout_metrics", {}) or {}
print("Best model      :", artifacts.get("best_model"))
print("Holdout RMSE    :", metrics.get("RMSE"))
print("Holdout MAE     :", metrics.get("MAE"))
print("Holdout R²      :", metrics.get("R2"))
print("Holdout CCC     :", metrics.get("CCC"))
print("Bathymetry TIF  :", artifacts.get("bathymetry_raster"))
print("Uncertainty TIF :", artifacts.get("uncertainty_raster"))
print("Model bundle    :", artifacts.get("model_path"))
print("Manifest        :", Path(pipeline.cfg.output_dir) / "deployment_manifest.json")

## All generated files

In [ ]:
out = Path(pipeline.cfg.output_dir)
for p in sorted(out.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1024:10.1f} KB  {p.relative_to(out)}")

## Show the publication figures inline

In [ ]:
from IPython.display import Image, display

plot_dir = Path(pipeline.cfg.output_dir) / "plots"
pngs = sorted(plot_dir.glob("*.png"))
if not pngs:
    print("No figures found in", plot_dir)
for png in pngs:
    print(png.name)
    display(Image(filename=str(png), width=520))

## Main metrics table (CSV preview)

In [ ]:
import pandas as pd

csv_path = Path(pipeline.cfg.output_dir) / "metrics" / "5_all_models_cv_vs_holdout.csv"
if csv_path.exists():
    display(pd.read_csv(csv_path).round(4))
else:
    print("CSV not found:", csv_path)

## Alternative ways to run (same Python code)

1. **Shell-style inside Jupyter** (the module strips Jupyter's injected
   `-f kernel.json` argument automatically):

   ```python
   !python sdb_framework_final.py --mode full --config config.yaml
   ```

   or

   ```python
   %run sdb_framework_final.py --mode full --config config.yaml
   ```

2. **One-call helper:**

   ```python
   from sdb_framework_final import run_sdb_for_files
   artifacts = run_sdb_for_files(
       excel_path="data/SDBNew Latlong.xlsx",
       raster_path="data/20220213_SDB_5bands.tif",
       output_dir="outputs/full_run",
   )
   ```

3. **Predict a new scene with the saved bundle:**

   ```python
   from sdb_framework_final import predict_new_scene
   predict_new_scene(
       "outputs/full_run/models/sdb_<name>_final.joblib",
       "new_scene.tif",
       "new_scene_bathymetry.tif",
   )
   ```